# Fine-tune DistilBERT on SMS messages

This notebook fine-tunes `distilbert-base-uncased` to classify SMS messages as
`benign` or `malicious`.

It uses the processed train/validation/test splits created in
`01_data_exploration.ipynb`, then saves the trained model and tokenizer to
`../models/distilbert-sms/`.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "models" / "distilbert-sms"

train_df = pd.read_csv(DATA_DIR / "train.csv")
validation_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(train_df.shape)
print(validation_df.shape)
print(test_df.shape)

train_df.head()

(3901, 2)
(558, 2)
(1115, 2)


,message,label
0,Welcome to UK-mobile-date this msg is FREE giv...,malicious
1,Theyre doing it to lots of places. Only hospit...,benign
2,Are your freezing ? Are you home yet ? Will yo...,benign
3,They released vday shirts and when u put it on...,benign
4,Ok lor then we go tog lor...\r\n,benign


In [4]:
from sms_classifier.model import (
    dataframe_to_dataset,
    load_tokenizer,
    tokenize_messages,
)

train_dataset = dataframe_to_dataset(train_df)
validation_dataset = dataframe_to_dataset(validation_df)
test_dataset = dataframe_to_dataset(test_df)

tokenizer = load_tokenizer()

tokenized_train_dataset = train_dataset.map(
    lambda examples: tokenize_messages(examples, tokenizer), batched=True
)
tokenized_validation_dataset = validation_dataset.map(
    lambda examples: tokenize_messages(examples, tokenizer), batched=True
)
tokenized_test_dataset = test_dataset.map(
    lambda examples: tokenize_messages(examples, tokenizer), batched=True
)

print(tokenized_train_dataset)
print(tokenized_train_dataset[0].keys())

Map: 100%|██████████| 1115/1115 [00:00<00:00, 6281.89 examples/s]

Dataset({
    features: ['message', 'label', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3901
})
dict_keys(['message', 'label', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])
